# 🔧 Tool Calling Tutorial: Data Preparation and Baseline

#### 📚 What you'll learn

This notebook walks through preparing tool calling data for fine-tuning and evaluation:

- How tool calling data is structured (OpenAI messages + tools format)
- How to prepare training, validation, and evaluation data splits
- How to upload data to NMP filesets for downstream use
- How well Nemotron Nano handles tool calling out-of-the-box (spoiler: it needs fine-tuning!)


### 📦 Imports

- `datasets` loads the xLAM dataset from Hugging Face.
- `nemo_microservices` provides the NMP platform client for filesets and evaluation.


### ⚡ Setup

Install the required dependencies and configure your environment. Update the values in [config.py](./config.py) with your NMP deployment URLs before running.


In [ ]:
%%capture
!pip install -r requirements.txt

In [ ]:
import os
import json
import random
from pprint import pprint
from typing import Any, Dict, List, Union

import numpy as np
from datasets import load_dataset
from nemo_microservices import NeMoMicroservices

from config import NEMO_URL, NIM_URL, WORKSPACE, BASE_MODEL, TRAINING_FILESET, EVAL_FILESET

### ⚙️ Initialize the NeMo Microservices client

- `NeMoMicroservices` is the main client for interacting with the NMP platform.
- The `workspace` parameter scopes all operations (filesets, evaluations, models, etc.).


In [ ]:
client = NeMoMicroservices(
    base_url=NEMO_URL,
    inference_base_url=NIM_URL,
    workspace=WORKSPACE,
)

In [ ]:
SEED = 1234
np.random.seed(SEED)
random.seed(SEED)

## 📥 Download the xLAM Dataset

- Salesforce [xLAM](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) is a large-scale function calling dataset with 60K examples.
- Each example contains a user query, available tools, and expected tool calls.

> 💡 **What is tool calling data?**
>
> - Tool calling (or function calling) trains a model to select the right API function and populate its arguments from a user's natural language query.
> - The training format uses `messages`, `tools`, and `tool_calls` in the [OpenAI format](https://platform.openai.com/docs/guides/function-calling).


In [ ]:
dataset = load_dataset("Salesforce/xlam-function-calling-60k")

example = dataset["train"][0]
pprint(example)

## 🔄 Convert to OpenAI Format

- The xLAM format needs conversion to OpenAI-compatible messages + tools format.
- This is the format NeMo Customizer expects for fine-tuning.


In [ ]:
def normalize_type(param_type: str) -> str:
    """Normalize Python type hints to OpenAI function spec types."""
    param_type = param_type.strip()

    if "," in param_type and "default" in param_type:
        param_type = param_type.split(",")[0].strip()
    if param_type.startswith("default="):
        return "string"

    param_type = param_type.replace(", optional", "").strip()

    if param_type.startswith("Callable"):
        return "string"
    if param_type.startswith(("Tuple", "List[", "Set")) or param_type in ("list", "set"):
        return "array"

    type_mapping = {
        "str": "string", "int": "integer", "float": "number",
        "bool": "boolean", "list": "array", "dict": "object",
        "List": "array", "Dict": "object", "Set": "array",
    }
    return type_mapping.get(param_type, "string")


def convert_tools_to_openai_spec(tools):
    """Convert xLAM tool definitions to OpenAI function spec format."""
    if isinstance(tools, str):
        tools = json.loads(tools)
    if not isinstance(tools, list):
        return []

    openai_tools = []
    for tool in tools:
        if not isinstance(tool, dict) or not isinstance(tool.get("parameters"), dict):
            continue

        properties = {}
        for name, info in tool["parameters"].items():
            if not isinstance(info, dict):
                continue
            param = {
                "description": info.get("description", ""),
                "type": normalize_type(info.get("type", "")),
            }
            default_value = info.get("default")
            if default_value is not None and default_value != "":
                param["default"] = default_value
            properties[name] = param

        openai_tools.append({
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": {"type": "object", "properties": properties},
            },
        })
    return openai_tools


def convert_tool_calls(xlam_tools):
    """Convert xLAM tool call format to OpenAI tool_calls format."""
    return [
        {"type": "function", "function": {"name": t["name"], "arguments": t.get("arguments", {})}}
        for t in json.loads(xlam_tools)
    ]


def convert_example(example):
    """Convert a single xLAM dataset example to OpenAI format."""
    obj = {"messages": [{"role": "user", "content": example["query"]}]}

    if example.get("tools"):
        obj["tools"] = convert_tools_to_openai_spec(example["tools"])

    assistant_message = {"role": "assistant", "content": ""}
    if example.get("answers"):
        assistant_message["tool_calls"] = convert_tool_calls(example["answers"])
    obj["messages"].append(assistant_message)

    return obj

- Let's verify the conversion on a single example.


In [ ]:
pprint(convert_example(example))

### 🏗️ Process the entire dataset

- Convert all examples to OpenAI format.
- Unlike the previous version of this tutorial, we keep multi-tool-call examples -- Nemotron models handle parallel tool calls.


In [ ]:
LIMIT_TOOL_PROPERTIES = 8

all_examples = []
for example in dataset["train"]:
    converted = convert_example(example)
    if converted is not None:
        all_examples.append(converted)

print(f"Converted {len(all_examples)} examples")

## ✂️ Split the Dataset

- 70% training, 15% validation, 15% test.
- We use a subset of 5000 examples to keep the tutorial fast.


In [ ]:
NUM_EXAMPLES = 5000
assert NUM_EXAMPLES <= len(all_examples)

sampled = random.sample(all_examples, NUM_EXAMPLES)

train_size = int(0.7 * len(sampled))
val_size = int(0.15 * len(sampled))

train_data = sampled[:train_size]
val_data = sampled[train_size:train_size + val_size]
test_data = sampled[train_size + val_size:]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

### Prepare evaluation format

- For evaluation, `tool_calls` must be a separate field (not nested inside `messages`).


In [ ]:
def convert_to_eval_format(entry):
    """Restructure an example for the evaluator: tool_calls as a top-level field."""
    for tool in entry.get("tools", []):
        if len(tool["function"]["parameters"]["properties"]) > LIMIT_TOOL_PROPERTIES:
            return None

    new_entry = {"messages": [], "tools": entry.get("tools", []), "tool_calls": []}
    for msg in entry["messages"]:
        if msg["role"] == "assistant" and "tool_calls" in msg:
            new_entry["tool_calls"] = msg["tool_calls"]
        else:
            new_entry["messages"].append(msg)
    return new_entry


test_data_eval = [r for entry in test_data if (r := convert_to_eval_format(entry)) is not None]
print(f"Evaluation examples: {len(test_data_eval)}")

### Save to local files


In [ ]:
def save_jsonl(filename, data):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, "w") as f:
        for entry in data:
            f.write(json.dumps(entry) + "\n")


DATA_ROOT = os.path.join(os.getcwd(), "data")

save_jsonl(os.path.join(DATA_ROOT, "training.jsonl"), train_data)
save_jsonl(os.path.join(DATA_ROOT, "validation.jsonl"), val_data)
save_jsonl(os.path.join(DATA_ROOT, "eval.jsonl"), test_data_eval)

print("Saved training.jsonl, validation.jsonl, eval.jsonl")

## 📤 Upload to NMP Filesets

- Filesets are the standard way to store datasets in NMP.
- Other services (Customizer, Evaluator) reference filesets by their workspace-scoped URN.


In [ ]:
def upload_fileset(client, name, description, local_path, remote_path):
    """Create a fileset and upload a file to it."""
    try:
        client.filesets.create(name=name, description=description)
        print(f"Created fileset: {name}")
    except Exception as e:
        if "409" in str(e):
            print(f"Fileset {name} already exists")
        else:
            raise

    with open(local_path, "rb") as f:
        client.filesets.upload_file(name=name, path=remote_path, body=f.read())
    print(f"Uploaded {local_path} -> {name}/{remote_path}")

In [ ]:
upload_fileset(
    client, TRAINING_FILESET, "xLAM training + validation data",
    os.path.join(DATA_ROOT, "training.jsonl"), "training/training.jsonl",
)

upload_fileset(
    client, TRAINING_FILESET, "xLAM training + validation data",
    os.path.join(DATA_ROOT, "validation.jsonl"), "validation/validation.jsonl",
)

upload_fileset(
    client, EVAL_FILESET, "xLAM evaluation data",
    os.path.join(DATA_ROOT, "eval.jsonl"), "data.jsonl",
)

## 🧪 Quick Baseline Evaluation

- Before fine-tuning, let's see how Nemotron Nano does on tool calling out-of-the-box.
- This gives us a baseline to compare against after fine-tuning.

> 💡 **Why start with a baseline?**
>
> - Measuring before and after is the foundation of the data flywheel.
> - A low baseline motivates the fine-tuning work ahead.


### 📏 Create a tool calling metric

- The custom `tool-calling` metric type gives two scores: `function_name_accuracy` and `function_name_and_args_accuracy`.


In [ ]:
try:
    client.evaluation.metrics.create(
        name="tool-calling-accuracy",
        type="tool-calling",
        reference="{{tool_calls}}",
    )
    print("Created metric: tool-calling-accuracy")
except Exception as e:
    if "409" in str(e):
        print("Metric tool-calling-accuracy already exists")
    else:
        raise

### 🚀 Launch baseline evaluation

- We evaluate Nemotron Nano on 50 examples from the test set.


In [ ]:
from nemo_microservices.types.evaluation import (
    MetricOfflineJobInputParam,
    EvaluationJobParamsParam,
)

eval_params = EvaluationJobParamsParam(
    inference={"model": BASE_MODEL},
    parallelism=16,
    limit_samples=50,
)

baseline_spec = MetricOfflineJobInputParam(
    metric=f"{WORKSPACE}/tool-calling-accuracy",
    dataset=f"{WORKSPACE}/{EVAL_FILESET}",
    params=eval_params,
)

baseline_job = client.evaluation.metric_jobs.create(spec=baseline_spec)
print(f"Launched baseline evaluation: {baseline_job.name}")

In [ ]:
from time import sleep, time

start = time()
while True:
    status = client.evaluation.metric_jobs.get_status(baseline_job.name)
    print(f"Status: {status.status} ({time() - start:.0f}s)")
    if status.status in ["completed", "failed", "cancelled", "error"]:
        break
    sleep(10)

### 📊 Baseline results


In [ ]:
results_list = client.evaluation.metric_jobs.results.list(baseline_job.name)

for result in results_list.data:
    print(f"Result: {result.result_name}")

print("\n--- Baseline Nemotron Nano Tool Calling Accuracy ---")
print("Without fine-tuning, expect low accuracy (~10-20%).")
print("This motivates the data generation and fine-tuning steps ahead.")

## ⏭️ Next Steps

Baseline accuracy is low — in the next notebook, we'll use **Data Designer** to generate high-quality synthetic training data to improve it.

- [2. Synthetic Data Generation with Data Designer](./2_data_designer.ipynb)
